In [ ]:
#| default_exp magickey

# MagicKey
> Passwordless auth combining magic links and passkeys

In [ ]:
#| export
from fasthtml.common import *
from webauthn import generate_authentication_options, generate_registration_options, verify_authentication_response, verify_registration_response, options_to_json
from webauthn.helpers import base64url_to_bytes, bytes_to_base64url
from webauthn.helpers.structs import AuthenticatorSelectionCriteria, ResidentKeyRequirement, UserVerificationRequirement
from json import loads
import secrets, time

In [ ]:
from fastcore.test import *
from starlette.testclient import TestClient

## Helpers

`_origin_kw` extracts the WebAuthn origin and relying party ID from the request. This is needed for both registration and authentication verification.


In [ ]:
#| export
def _origin_kw(req):
    host = req.url.hostname
    scheme = 'http' if host in ('localhost', '127.0.0.1') else 'https'
    return dict(expected_origin=f'{scheme}://{req.url.netloc}', expected_rp_id=host)

## MagicKey class

Follows the same pattern as `OAuth`: wires up beforeware and backend routes in `__init__`, with overridable methods for app-specific behaviour.

The `MagicKey` class sets up beforeware (redirects unauthenticated users to the login page) and registers all backend routes. The developer subclasses it and overrides `get_auth` (what URL to redirect to after login) and the passkey storage methods.

In [ ]:
#| export
class MagicKey:
    def __init__(self, app, send_email, skip=None, login_path='/login', logout_path='/logout'):
        if not skip: skip = [login_path, '/verify_magiclink', '/send_magic_link', '/request_passkey_auth',
                             '/verify_passkey_auth', '/setup_passkey', '/request_passkey_reg',
                             '/finish_passkey_reg', '/skip_passkey_reg']
        self.magiclink_db = {}
        store_attr()

        async def before(req, session):
            if 'auth' not in req.scope: req.scope['auth'] = session.get('auth')
            if not req.scope['auth']: return RedirectResponse(login_path, status_code=303)
        app.before.append(Beforeware(before, skip=skip))

        @app.get(logout_path)
        def logout(session):
            session.pop('auth', None)
            return RedirectResponse(login_path, status_code=303)

        @app.post('/send_magic_link')
        def send_magic_link(email: str, req):
            token = secrets.token_urlsafe(32)
            self.magiclink_db[token] = dict(email=email, timestamp=time.time(), used=False)
            scheme = 'http' if req.url.hostname in ('localhost', '127.0.0.1') else 'https'
            magic_url = f'{scheme}://{req.url.netloc}/verify_magiclink?token={token}'
            return self.send_email(email, magic_url)

        @app.get('/verify_magiclink')
        def verify_magiclink(token: str, session, req):
            link = self.magiclink_db.get(token)
            if not link or link['used'] or time.time() - link['timestamp'] > 3600: return P('Invalid or expired link')
            link['used'] = True
            return self.after_magiclink_verify(link['email'], session, req)

        @app.post('/request_passkey_auth')
        def request_passkey_auth(session, req):
            auth_opts = generate_authentication_options(
                rp_id=req.url.hostname, user_verification=UserVerificationRequirement.PREFERRED)
            session['auth_challenge'] = bytes_to_base64url(auth_opts.challenge)
            return Script("""
                SimpleWebAuthnBrowser.startAuthentication({ optionsJSON: %s }).then(r => {
                    htmx.ajax('POST', '/verify_passkey_auth', { values: r, target:'#result' });
                });""" % options_to_json(auth_opts))

        @app.post('/verify_passkey_auth')
        def verify_passkey_auth(response: str, id: str, data: dict, session, req):
            challenge_b64 = session.pop('auth_challenge', None)
            if not challenge_b64: return HttpHeader('HX-Redirect', f'{login_path}?error=no_challenge')
            cred_id = base64url_to_bytes(id)
            stored = self.get_passkey(cred_id)
            if not stored: return HttpHeader('HX-Redirect', f'{login_path}?error=passkey_not_found')
            data['response'] = loads(response)
            try:
                res = verify_authentication_response(
                    credential_public_key=stored['public_key'], credential_current_sign_count=stored['sign_count'],
                    credential=data, expected_challenge=base64url_to_bytes(challenge_b64), require_user_verification=True, **_origin_kw(req))
            except Exception: return HttpHeader('HX-Redirect', f'{login_path}?error=passkey_failed')
            self.update_passkey(cred_id, res.new_sign_count)
            session['auth'] = stored['email']
            return self._do_auth(stored['email'], session, htmx=True)

        @app.post('/request_passkey_reg')
        def request_passkey_reg(session, req):
            email = session.get('pending_email')
            if not email: return RedirectResponse(f'{login_path}?error=no_pending_reg', status_code=303)
            uid = base64url_to_bytes(email)
            selec = AuthenticatorSelectionCriteria(resident_key=ResidentKeyRequirement.REQUIRED, require_resident_key=True)
            opts = generate_registration_options(
                rp_id=req.url.hostname, rp_name='MagicKey', user_name=email, user_id=uid, authenticator_selection=selec)
            session['reg_challenge'] = bytes_to_base64url(opts.challenge)
            return Script("""
                SimpleWebAuthnBrowser.startRegistration({ optionsJSON: %s }).then(r => {
                    htmx.ajax('POST', '/finish_passkey_reg', { values: r, target:'#result' });
                });""" % options_to_json(opts))

        @app.post('/finish_passkey_reg')
        def finish_passkey_reg(response: str, data: dict, session, req):
            email = session.pop('pending_email', None)
            if not email: return RedirectResponse(f'{login_path}?error=no_pending_reg', status_code=303)
            challenge_b64 = session.pop('reg_challenge', None)
            if not challenge_b64: return RedirectResponse('/setup_passkey?error=no_challenge', status_code=303)
            data['response'] = loads(response)
            try:
                res = verify_registration_response(
                    credential=data, expected_challenge=base64url_to_bytes(challenge_b64), require_user_verification=True, **_origin_kw(req))
            except Exception: return RedirectResponse('/setup_passkey?error=reg_failed', status_code=303)
            self.save_passkey(res.credential_id, email, res.credential_public_key, res.sign_count)
            session['auth'] = email
            return self._do_auth(email, session, htmx=True)

        @app.get('/skip_passkey_reg')
        def skip_passkey_reg(session):
            email = session.pop('pending_email', None)
            if not email: return RedirectResponse(login_path, status_code=303)
            session['auth'] = email
            return self._do_auth(email, session, htmx=False)

    def _do_auth(self, email, session, htmx=False):
        url = self.get_auth(email, session)
        if htmx: return HttpHeader('HX-Redirect', url)
        return RedirectResponse(url, status_code=303)

    def get_auth(self, email, session): raise NotImplementedError()
    def has_passkey(self, email): return False
    def get_passkey(self, credential_id): return None
    def save_passkey(self, credential_id, email, public_key, sign_count): raise NotImplementedError()
    def update_passkey(self, credential_id, sign_count): raise NotImplementedError()

    def after_magiclink_verify(self, email, session, req):
        if self.has_passkey(email):
            session['auth'] = email
            return self._do_auth(email, session, htmx=False)
        session['pending_email'] = email
        return RedirectResponse('/setup_passkey', status_code=303)

### Overridable methods

| Method | Purpose | Default |
|--------|---------|---------|
| `get_auth(email, session)` | Return URL to redirect to after successful auth | `NotImplementedError` |
| `has_passkey(email)` | Check if user has a registered passkey | `False` |
| `get_passkey(credential_id)` | Look up passkey by credential ID, return dict with `email`, `public_key`, `sign_count` | `None` |
| `save_passkey(credential_id, email, public_key, sign_count)` | Store a new passkey | `NotImplementedError` |
| `update_passkey(credential_id, sign_count)` | Update sign count after auth | `NotImplementedError` |
| `after_magiclink_verify(email, session, req)` | Called after magic link verified; default offers passkey registration | redirects to `/setup_passkey` |

## Example app

Here's a minimal app using `MagicKey` with in-memory passkey storage. The developer provides the login page, setup page, and storage — MagicKey handles the auth routes.

In [ ]:
passkey_store = {}

class Auth(MagicKey):
    def get_auth(self, email, session): return '/'
    def has_passkey(self, email): return any(v['email'] == email for v in passkey_store.values())
    def get_passkey(self, credential_id): return passkey_store.get(credential_id)
    def save_passkey(self, credential_id, email, public_key, sign_count):
        passkey_store[credential_id] = dict(email=email, public_key=public_key, sign_count=sign_count)
    def update_passkey(self, credential_id, sign_count): passkey_store[credential_id]['sign_count'] = sign_count

app, rt = fast_app(secret_key='test')
def _dev_send_email(email, url): return P(f'Magic link for {email}: ', A(url, href=url))

mk = Auth(app, send_email=_dev_send_email)

simplewebauthn = Script(src='https://cdn.jsdelivr.net/npm/@simplewebauthn/browser@13.1.0/dist/bundle/index.umd.min.js')

@rt('/')
def home(auth): return P(f'Hello {auth}!'), A('Log out', href='/logout')

@rt('/login')
def login():
    return Titled('Sign In', simplewebauthn,
        Button('Sign in with Passkey', hx_post='/request_passkey_auth', target_id='scripts'),
        Hr(),
        Form(method='post', action='/send_magic_link')(
            Input(name='email', type='email', placeholder='you@example.com'),
            Button('Send Magic Link')),
        Div(id='scripts'), Div(id='result'))

@rt('/setup_passkey')
def setup_passkey():
    return Titled('Set Up Passkey', simplewebauthn,
        P('Set up a passkey for faster logins next time?'),
        Button('Register Passkey', hx_post='/request_passkey_reg', target_id='scripts'),
        A('Skip', href='/skip_passkey_reg'),
        Div(id='scripts'), Div(id='result'))

## Tests

We can test the non-WebAuthn parts using FastHTML's test client. The beforeware, magic link flow, and session management are all testable without a browser.

In [ ]:
cli = TestClient(app)

r = cli.get('/', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login')

r = cli.get('/login')
test_eq(r.status_code, 200)

In [ ]:
r = cli.post('/send_magic_link', data={'email': 'expired@test.com'})
token = [k for k in mk.magiclink_db.keys()][-1]
mk.magiclink_db[token]['timestamp'] = time.time() - 3601  # older than 1hr
r = cli.get(f'/verify_magiclink?token={token}', follow_redirects=True)
test_eq(r.text.find('Invalid or expired') >= 0, True)


In [ ]:
r = cli.post('/send_magic_link', data={'email': 'test@example.com'})
test_eq(r.status_code, 200)
token = list(mk.magiclink_db.keys())[-1]

r = cli.get(f'/verify_magiclink?token={token}', follow_redirects=False)
test_eq(r.status_code, 303)
test(r.headers['location'], '/setup_passkey', operator.contains)

In [ ]:
r = cli.get('/skip_passkey_reg', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/')

r = cli.get('/')
test_eq(r.status_code, 200)
test(r.text, 'test@example.com', operator.contains)

In [ ]:
r = cli.get('/logout', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login')

r = cli.get('/', follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login')

Lets add some tests for the unhappy paths

In [ ]:
r = cli.get(f'/verify_magiclink?token={token}')
test(r.text, 'Invalid or expired', operator.contains)

r = cli.get('/verify_magiclink?token=bogus')
test(r.text, 'Invalid or expired', operator.contains)

In [ ]:
r = cli.post('/verify_passkey_auth', data={'response': '{}', 'id': 'fake', 'data': '{}'}, follow_redirects=False)
test_eq(r.status_code, 200)
test_eq(r.headers['hx-redirect'], '/login?error=no_challenge')

In [ ]:
r = cli.post('/request_passkey_auth', headers={'hx-request': 'true'})
r = cli.post('/verify_passkey_auth', data={'response': '{}', 'id': 'fake', 'data': '{}'}, follow_redirects=False)
test_eq(r.status_code, 200)
test_eq(r.headers['hx-redirect'], '/login?error=passkey_not_found')

In [ ]:
# Test C: finish_passkey_reg with no pending_email → redirect, not P()
r = cli.post('/finish_passkey_reg', data={'response': '{}', 'data': '{}'}, follow_redirects=False)
test_eq(r.status_code, 303)
test_eq(r.headers['location'], '/login?error=no_pending_reg')

## Export -

In [ ]:
#| hide
#| eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()